In [2]:
# 1. Install Kaggle library and upload your API token
!pip install -q kaggle
from google.colab import files
uploaded = files.upload() # Choose your kaggle.json file here

# 2. Move the kaggle.json file to the correct hidden directory
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Download a standard 5-class weather dataset
# (This downloads the dataset directly to Colab's high-speed servers)
!kaggle datasets download -d pratik2901/multiclass-weather-dataset
!unzip -q multiclass-weather-dataset.zip -d weather_data/

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/pratik2901/multiclass-weather-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 91.4M/91.4M [00:04<00:00, 21.1MB/s]



In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50

# 1. Load the Data (Assuming your data is unzipped in 'weather_data/')
# We use image_dataset_from_directory to pull images in batches so we don't crash the RAM
DATA_DIR = "weather_data/Multi-class Weather Dataset"
BATCH_SIZE = 32
IMG_SIZE = (224, 224) # Standard input size for ResNet50

train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical' # Mutually exclusive weather classes
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# 2. Data Augmentation (To prevent overfitting)
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.2),
])

Found 1125 files belonging to 4 classes.
Using 900 files for training.
Found 1125 files belonging to 4 classes.
Using 225 files for validation.


In [9]:
# 1. Load Pre-trained ResNet50 (without the final ImageNet classification head)
# We also load the 'imagenet' weights.
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Freeze the base model so we don't destroy its pre-trained features
base_model.trainable = False

# 3. Build our Custom Classification Head
inputs = tf.keras.Input(shape=(224, 224, 3))

# Important: ResNet50 requires a specific preprocessing function for its pixels
# rather than our standard 1./255 division.
x = data_augmentation(inputs)
x = tf.keras.applications.resnet50.preprocess_input(x)

# Pass the augmented/processed images through the frozen ResNet base
x = base_model(x, training=False)

# Replaces the old Flatten layer to prevent the 11-million parameter bottleneck
x = layers.GlobalAveragePooling2D()(x)

# Add a dropout layer to prevent overfitting on the new head
x = layers.Dropout(0.3)(x)

# Final decision layer for 4 weather classes
outputs = layers.Dense(4, activation='softmax')(x)

# Compile the final model
model = tf.keras.Model(inputs, outputs)

# We use Adam optimizer and Categorical Cross-Entropy for multi-class classification
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 224, 224,  │          0 │ input_layer_4[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 224, 224)  │          0 │ sequential_1[1][… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 224, 224)  │          0 │ sequential_1[1][… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 224, 224)  │          0 │ sequential_1[1][… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 224, 224,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 224, 224,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add_1[0][0]       │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 2048)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 4)         │      8,196 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,595,908 (90.01 MB)

 Trainable params: 8,196 (32.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [10]:
# Stop training automatically if validation loss stops improving
early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=[early_stopper]
)

Epoch 1/15
29/29 ━━━━━━━━━━━━━━━━━━━━ 222s 7s/step - accuracy: 0.6778 - loss: 0.8379 - val_accuracy: 0.9067 - val_loss: 0.2751
Epoch 2/15
29/29 ━━━━━━━━━━━━━━━━━━━━ 191s 7s/step - accuracy: 0.9222 - loss: 0.2348 - val_accuracy: 0.9289 - val_loss: 0.1648
Epoch 3/15
29/29 ━━━━━━━━━━━━━━━━━━━━ 196s 7s/step - accuracy: 0.9422 - loss: 0.1588 - val_accuracy: 0.9511 - val_loss: 0.1390
Epoch 4/15
29/29 ━━━━━━━━━━━━━━━━━━━━ 190s 7s/step - accuracy: 0.9578 - loss: 0.1340 - val_accuracy: 0.9600 - val_loss: 0.1083
Epoch 5/15
29/29 ━━━━━━━━━━━━━━━━━━━━ 194s 7s/step - accuracy: 0.9589 - loss: 0.1131 - val_accuracy: 0.9689 - val_loss: 0.1019
Epoch 6/15
29/29 ━━━━━━━━━━━━━━━━━━━━ 192s 7s/step - accuracy: 0.9656 - loss: 0.1108 - val_accuracy: 0.9556 - val_loss: 0.0994
Epoch 7/15
29/29 ━━━━━━━━━━━━━━━━━━━━ 203s 7s/step - accuracy: 0.9689 - loss: 0.0983 - val_accuracy: 0.9689 - val_loss: 0.0915
Epoch 8/15
29/29 ━━━━━━━━━━━━━━━━━━━━ 201s 7s/step - accuracy: 0.9722 - loss: 0.0996 - val_accuracy: 0.9689 - v

In [13]:
# 1. Install the Gradio library silently
!pip install gradio -q

import gradio as gr
import numpy as np
import tensorflow as tf

# 2. Dynamically grab your 4 actual class names from the dataset
class_names = train_dataset.class_names

# 3. Define the prediction function
def predict_weather(image):
    # Resize the uploaded image to match ResNet50's expected (224, 224) input
    img = tf.image.resize(image, (224, 224))
    img = tf.expand_dims(img, 0) # Create a batch of 1

    # Make the prediction
    predictions = model.predict(img)

    # Return a dictionary of class names and their probabilities for Gradio to display
    return {class_names[i]: float(predictions[0][i]) for i in range(len(class_names))}

# 4. Create the beautiful drag-and-drop web interface
interface = gr.Interface(
    fn=predict_weather,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=4),
    title="🌤️ ResNet50 Weather Classifier",
    description="Upload a photo of the sky or weather, and the model will predict the current conditions!"
)

# 5. Launch the UI right here in Colab
interface.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bcfc1c77e60055ff60.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
# Save the model to an HDF5 file (.h5 or .keras)
model.save("02_resnet50_weather_classifier.keras")
print("Model successfully saved!")

Model successfully saved!
